# Chuẩn bị dữ liệu Seoul Bike Sharing Demand
Nguồn: UCI Machine Learning Repository, DOI: 10.24432/C5F62R. Tệp gốc được giữ nguyên; các thay đổi được ghi ở những ô mã bên dưới. Đặt notebook cùng thư mục với `SeoulBikeData_raw.csv`.

In [5]:
from pathlib import Path
import pandas as pd

data_dir = Path.cwd()
raw_path = data_dir / 'SeoulBikeData_raw.csv'
clean_path = data_dir / 'SeoulBikeData_clean.csv'
df = pd.read_csv(raw_path, encoding='cp1252')  # ký tự ° trong tiêu đề cột
print('Kích thước:', df.shape)
display(df.head())
display(df.dtypes.to_frame('Kiểu dữ liệu'))

Kích thước: (8760, 14)


,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


,Kiểu dữ liệu
Date,str
Rented Bike Count,int64
Hour,int64
Temperature(°C),float64
Humidity(%),int64
Wind speed (m/s),float64
Visibility (10m),int64
Dew point temperature(°C),float64
Solar Radiation (MJ/m2),float64
Rainfall(mm),float64


In [6]:
print('Số ô khuyết:', int(df.isna().sum().sum()))

print('Số dòng trùng:', int(df.duplicated().sum()))

print('Cặp Date–Hour trùng:', int(df.duplicated(['Date', 'Hour']).sum()))

display(pd.DataFrame({'Kiểu dữ liệu': df.dtypes.astype(str), 'Số giá trị duy nhất': df.nunique(), 'Số ô khuyết': df.isna().sum()}))

display(df.select_dtypes(include='number').agg(['min', 'max']).T)

for col in ['Seasons', 'Holiday', 'Functioning Day']:
    print(col, df[col].value_counts().to_dict())
dates = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='raise')
assert len(df) == 8760 and len(df.columns) == 14
assert not df.isna().any().any() and not df.duplicated().any()
assert not df.duplicated(['Date', 'Hour']).any()
assert df.groupby('Date')['Hour'].nunique().eq(24).all()

print('Giai đoạn:', dates.min().date(), 'đến', dates.max().date())

Số ô khuyết: 0
Số dòng trùng: 0
Cặp Date–Hour trùng: 0


,Kiểu dữ liệu,Số giá trị duy nhất,Số ô khuyết
Date,str,365,0
Rented Bike Count,int64,2166,0
Hour,int64,24,0
Temperature(°C),float64,546,0
Humidity(%),int64,90,0
Wind speed (m/s),float64,65,0
Visibility (10m),int64,1789,0
Dew point temperature(°C),float64,556,0
Solar Radiation (MJ/m2),float64,345,0
Rainfall(mm),float64,61,0


,min,max
Rented Bike Count,0.0,3556.00
Hour,0.0,23.00
Temperature(°C),-17.8,39.40
Humidity(%),0.0,98.00
Wind speed (m/s),0.0,7.40
Visibility (10m),27.0,2000.00
Dew point temperature(°C),-30.6,27.20
Solar Radiation (MJ/m2),0.0,3.52
Rainfall(mm),0.0,35.00
Snowfall (cm),0.0,8.80


Seasons {'Spring': 2208, 'Summer': 2208, 'Autumn': 2184, 'Winter': 2160}
Holiday {'No Holiday': 8328, 'Holiday': 432}
Functioning Day {'Yes': 8465, 'No': 295}
Giai đoạn: 2017-12-01 đến 2018-11-30


## Kiểm tra giá trị 0 và giá trị lớn
Các giờ hệ thống không hoạt động có số lượt thuê bằng 0. Các giá trị thuê cao được kiểm tra bằng quy tắc IQR để nhận diện, không tự động xóa vì có thể phản ánh nhu cầu thực.

In [7]:
y = df['Rented Bike Count']
q1, q3 = y.quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print('Số giờ không hoạt động:', int(df['Functioning Day'].eq('No').sum()))

print('Số giờ có 0 lượt thuê:', int(y.eq(0).sum()))

print('0 lượt thuê nhưng hệ thống hoạt động:', int((y.eq(0) & df['Functioning Day'].eq('Yes')).sum()))

print('Ngưỡng IQR phía trên:', upper, '; số giá trị vượt ngưỡng:', int(y.gt(upper).sum()))

Số giờ không hoạt động: 295
Số giờ có 0 lượt thuê: 295
0 lượt thuê nhưng hệ thống hoạt động: 0
Ngưỡng IQR phía trên: 2376.625 ; số giá trị vượt ngưỡng: 158


In [8]:
clean = df.copy()
clean['Date'] = dates.dt.strftime('%Y-%m-%d')
clean['Month'] = dates.dt.month
clean['DayOfWeek'] = dates.dt.dayofweek  # Thứ Hai=0, Chủ nhật=6
clean['IsWeekend'] = dates.dt.dayofweek.ge(5).astype(int)
assert clean.shape == (8760, 17)
assert clean['Date'].str.match(r'\d{4}-\d{2}-\d{2}').all()
clean.to_csv(clean_path, index=False, encoding='utf-8-sig')
print('Đã lưu:', clean_path.name, '; số dòng:', len(clean), '; số cột:', len(clean.columns))
print('Không xóa dòng, không điền giá trị, không loại điểm thuê cao.')

Đã lưu: SeoulBikeData_clean.csv ; số dòng: 8760 ; số cột: 17
Không xóa dòng, không điền giá trị, không loại điểm thuê cao.
